# Bölüm 5/7 — Esik Optimizasyonu Ve Hata Analizi



## Önceki bölümden devam
Bu hücre, `4_...` bölümünde kaydedilen tüm değişkenleri ve tanımlı fonksiyonları geri yükler.

In [ ]:
!pip install dill --quiet
import dill
dill.load_session('checkpoint_4.pkl')
print('Önceki bölümün oturumu yüklendi.')

## Karar Eşiği (Threshold) Optimizasyonu

Şimdiye kadarki tüm modeller varsayılan 0.5 karar eşiğini (ya da `decision_function` için 0 sınırını) kullanıyordu. Ama dengesiz veri setlerinde bu eşik nadiren optimaldir. **Önemli metodolojik nokta:** en iyi eşiği doğrudan test setinde aramak, test setini bir hiperparametre gibi kullanmak anlamına gelir ve sonucu iyimser gösterebilir (veri sızıntısı). Bunun yerine eşik, eğitim verisinin `cross_val_predict` ile elde edilen 5-katlı çapraz-doğrulanmış (out-of-fold) tahminlerinde aranır; test seti yalnızca bulunan sabit eşikle **tek seferlik nihai değerlendirme** için kullanılır. Model destekliyorsa `predict_proba`, desteklemiyorsa (örn. Linear SVM) `decision_function` kullanılır.

In [ ]:
import numpy as np
from sklearn.base import clone
from sklearn.model_selection import cross_val_predict

# ---- 1) Esik SADECE egitim verisinin capraz-dogrulanmis (out-of-fold) tahminlerinde araniyor ----
#     Test setine bu asamada hic dokunulmuyor.
if en_iyi_model_adi == "Optimized XGBoost":
    oof_proba = cross_val_predict(
        clone(en_iyi_model), X_train_tfidf, y_train_xgb, cv=5, method="predict_proba", n_jobs=-1
    )[:, 0]
    y_train_no_thr = (y_train_xgb == 0).astype(int)
    varsayilan_esik = 0.5
elif hasattr(en_iyi_model, "predict_proba"):
    oof_proba_full = cross_val_predict(
        clone(en_iyi_model), X_train_tfidf, y_train, cv=5, method="predict_proba", n_jobs=-1
    )
    classes = list(en_iyi_model.classes_)
    no_idx = classes.index("no")
    oof_proba = oof_proba_full[:, no_idx]
    y_train_no_thr = (y_train == "no").astype(int)
    varsayilan_esik = 0.5
else:
    # LinearSVC gibi sadece decision_function sunan modeller icin
    oof_ham = cross_val_predict(
        clone(en_iyi_model), X_train_tfidf, y_train, cv=5, method="decision_function", n_jobs=-1
    )
    classes = list(en_iyi_model.classes_)
    oof_proba = -oof_ham if classes[1] == "yes" else oof_ham
    y_train_no_thr = (y_train == "no").astype(int)
    varsayilan_esik = 0.0  # decision_function=0 sinir noktasidir

precision_no, recall_no, thresholds = precision_recall_curve(y_train_no_thr, oof_proba)
f1_no = 2 * precision_no * recall_no / (precision_no + recall_no + 1e-12)
en_iyi_thr_idx = int(np.argmax(f1_no[:-1]))  # thresholds dizisi 1 eksik uzunlukta
en_iyi_esik = thresholds[en_iyi_thr_idx]

print(f"Model: {en_iyi_model_adi}")
print("Esik, egitim verisinin 5-katli capraz-dogrulanmis (out-of-fold) tahminleri uzerinde araniyor")
print("(test seti bu asamada kullanilmiyor).")
print(f"'no' sinifi F1'ini maksimize eden esik (egitim CV): {en_iyi_esik:.4f}  (varsayilan: {varsayilan_esik})")
print(f"Egitim CV'sindeki bu esikte -> Precision: {precision_no[en_iyi_thr_idx]:.4f}  "
      f"Recall: {recall_no[en_iyi_thr_idx]:.4f}  F1: {f1_no[en_iyi_thr_idx]:.4f}")

# ---- 2) Bulunan sabit esik, TEST setinde SADECE BIR KEZ nihai degerlendirme icin kullanilir ----
if en_iyi_model_adi == "Optimized XGBoost":
    skor_no_test = en_iyi_model.predict_proba(X_test_tfidf)[:, 0]
    y_true_no_thr = (en_iyi_y_true == 0).astype(int)
elif hasattr(en_iyi_model, "predict_proba"):
    classes = list(en_iyi_model.classes_)
    no_idx = classes.index("no")
    skor_no_test = en_iyi_model.predict_proba(X_test_tfidf)[:, no_idx]
    y_true_no_thr = (en_iyi_y_true == "no").astype(int)
else:
    classes = list(en_iyi_model.classes_)
    ham_skor = en_iyi_model.decision_function(X_test_tfidf)
    skor_no_test = -ham_skor if classes[1] == "yes" else ham_skor
    y_true_no_thr = (en_iyi_y_true == "no").astype(int)

varsayilan_tahmin_no = (skor_no_test >= varsayilan_esik).astype(int)
yeni_tahmin_no = (skor_no_test >= en_iyi_esik).astype(int)

print("\n--- TEST setinde varsayilan esik ile classification report ('no' pozitif kabul edilerek) ---")
print(classification_report(y_true_no_thr, varsayilan_tahmin_no, target_names=["diger", "no"]))
print("--- TEST setinde egitim-CV'siyle bulunan esikle classification report ('no' pozitif kabul edilerek) ---")
print(classification_report(y_true_no_thr, yeni_tahmin_no, target_names=["diger", "no"]))


**Yorum:** Eşik, eğitim verisinin çapraz-doğrulanmış tahminlerinde bulunduğu ve test setine yalnızca tek seferlik nihai değerlendirme için uygulandığı için, buradaki karşılaştırma test setini "görmeden" elde edilmiş adil bir tahmindir. Optimize edilmiş eşikteki F1, varsayılan eşikten belirgin şekilde yüksekse, modelin gerçek dünyada 0.5 yerine bu yeni eşikle devreye alınması önerilir. Fark küçükse, model zaten dengesizliğe karşı (GridSearchCV ile bulunan `class_weight`/`scale_pos_weight` sayesinde) makul ölçüde dayanıklı demektir. Not: `cross_val_predict` modelinizi 5 kez daha eğittiği için bu hücre önceki sürüme göre biraz daha uzun sürebilir.

##  Model Hata Analizi

Herhangi bir modelin tahminlerini analiz eden genel amaçlı bir fonksiyon (`hata_analizi_yap`) tanımlanır: yanlış tahmin sayısını, yanlış sınıflandırılan metinleri, ve False Positive/False Negative örneklerini raporlar. Bu fonksiyon, kod tekrarını önlemek amacıyla her model için tekrar tekrar çağrılabilir.

In [ ]:
def hata_analizi_yap(model_adi, y_pred):
    hatali_mask = (y_test.values != y_pred)
    hata_df = pd.DataFrame({
        "clean_text": X_test_text.values[hatali_mask],
        "gercek": y_test.values[hatali_mask],
        "tahmin": y_pred[hatali_mask]
    })

    # 1) Yanlis tahminlerin sayisi
    print(f"=== {model_adi} ===")
    print("Yanlış tahmin sayısı:", len(hata_df))

    # 2) Yanlis siniflandirilan metinler
    print("\nYanlış sınıflandırılan metinler:")
    display(hata_df[["clean_text", "gercek", "tahmin"]].head(10))

    # 3) False Positive ornek (gercekte no, tahmin yes)
    fp = hata_df[(hata_df["gercek"] == "no") & (hata_df["tahmin"] == "yes")]
    print("\nFalse Positive örnek (gerçekte haber, dezenformasyon sanılmış):")
    display(fp[["clean_text"]].head(5))

    # 4) False Negative ornek (gercekte yes, tahmin no)
    fn = hata_df[(hata_df["gercek"] == "yes") & (hata_df["tahmin"] == "no")]
    print("\nFalse Negative örnek (gerçekte dezenformasyon, haber sanılmış):")
    display(fn[["clean_text"]].head(5))

    return hata_df

### SVM

`hata_analizi_yap` fonksiyonu, optimize edilmiş Linear SVM modelinin tahminlerine uygulanır.

In [ ]:
hata_svm = hata_analizi_yap("Linear SVM (optimize)", y_pred_best_svm)

### Random Forest

`hata_analizi_yap` fonksiyonu, optimize edilmiş Random Forest modelinin tahminlerine uygulanır.

In [ ]:
hata_rf = hata_analizi_yap("Random Forest (optimize)", y_pred_best_rf)

### XGBoost

XGBoost'un hedef değişkeni 0/1 olarak kodlandığından ("no"/"yes" değil), bu model için ayrı bir hata analizi fonksiyonu (`hata_analizi_yap_xgb`) tanımlanıp optimize edilmiş XGBoost modeline uygulanır.

In [ ]:
def hata_analizi_yap_xgb(model_adi, y_pred, y_true, X_metin):
    hatali_mask = (y_true.values != y_pred)
    hata_df = pd.DataFrame({
        "clean_text": X_metin.values[hatali_mask],
        "gercek": y_true.values[hatali_mask],
        "tahmin": y_pred[hatali_mask]
    })
    print(f"=== {model_adi} ===")
    print("Yanlış tahmin sayısı:", len(hata_df))
    display(hata_df.head(10))
    fp = hata_df[(hata_df["gercek"]==0)&(hata_df["tahmin"]==1)]
    fn = hata_df[(hata_df["gercek"]==1)&(hata_df["tahmin"]==0)]
    print("\nFalse Positive örnek:")
    display(fp[["clean_text"]].head(5))
    print("\nFalse Negative örnek:")
    display(fn[["clean_text"]].head(5))
    return hata_df

hata_xgb = hata_analizi_yap_xgb("XGBoost (optimize)", y_pred_best_xgb, y_test_xgb, X_test_text)

## 20. Özellik Önem Analizi

## Linear SVM

Linear SVM modelinin `coef_` (katsayı) değerleri incelenir: pozitif katsayılı kelimeler "yes" (dezenformasyon), negatif katsayılı kelimeler "no" (gerçek haber) sınıfını işaret eder. SVM doğrusal bir model olduğu için bu katsayılar doğrudan yorumlanabilir.

In [ ]:
import numpy as np

coefs = best_svm.coef_[0]
order = np.argsort(coefs)

print("Linear SVM - 'yes' (dezenformasyon) yönünde en güçlü 12 özellik:")
for i in order[-12:][::-1]:
    print(f"  {feature_names[i]:20s} {coefs[i]:.3f}")

print("\nLinear SVM - 'no' (gerçek haber) yönünde en güçlü 12 özellik:")
for i in order[:12]:
    print(f"  {feature_names[i]:20s} {coefs[i]:.3f}")

## Random Forest

Random Forest modelinin `feature_importances_` değerleri incelenir. Bu metrik yönsüzdür (sadece genel ayırt edicilik gücünü gösterir, hangi sınıfa işaret ettiğini değil).

In [ ]:
imp_rf = best_rf.feature_importances_
order_rf = np.argsort(imp_rf)[::-1]

print("Random Forest - en önemli 15 özellik (YÖNSÜZ - sadece genel ayırt edicilik):")
for i in order_rf[:15]:
    print(f"  {feature_names[i]:20s} {imp_rf[i]:.4f}")

## XGBoost

En iyi model olan XGBoost'un (`best_xgb`) `feature_importances_` değerleri incelenir. Random Forest'takiyle aynı şekilde bu metrik de yönsüzdür (bir kelimenin genel ayırt edicilik gücünü gösterir, "yes" mi "no" mu yönünde olduğunu değil). Varsayılan `importance_type="gain"` kullanılır: bir özelliğin ağaçlardaki bölünmelerde ortalama ne kadar performans kazancı (impurity azalması) sağladığını ölçer.

In [ ]:
imp_xgb = best_xgb.feature_importances_
order_xgb = np.argsort(imp_xgb)[::-1]

print("XGBoost - en önemli 15 özellik (YÖNSÜZ - sadece genel ayırt edicilik, importance_type='gain'):")
for i in order_xgb[:15]:
    print(f"  {feature_names[i]:20s} {imp_xgb[i]:.4f}")


**Yorum:** Bu sıralamanın Linear SVM ve Random Forest'ta bulunan en güçlü kelimelerle (örn. "propaganda", "fake", "claims", "allegedly") ne kadar örtüştüğüne bakmak, modeller arasında tutarlı bir sinyal olup olmadığını gösterir — üç farklı model ailesi (lineer, bagging, boosting) aynı kelimelere yöneliyorsa, bu bulgunun veri setine özgü bir yapaylık değil, gerçek bir dilsel örüntü olduğuna dair güveni artırır.

## Multinomial NB

Multinomial Naive Bayes'in `feature_log_prob_` değerleri üzerinden, "yes" ve "no" sınıfları arasındaki log-olasılık farkı hesaplanarak en ayırt edici kelimeler (SVM'deki katsayı mantığına benzer şekilde, yönlü olarak) belirlenir.

In [ ]:
yes_idx = list(mnb_model.classes_).index("yes")
no_idx = list(mnb_model.classes_).index("no")
diff_mnb = mnb_model.feature_log_prob_[yes_idx] - mnb_model.feature_log_prob_[no_idx]
order_mnb = np.argsort(diff_mnb)

print("Multinomial NB - 'yes' yönünde en güçlü 12 özellik:")
for i in order_mnb[-12:][::-1]:
    print(f"  {feature_names[i]:20s} {diff_mnb[i]:.3f}")

print("\nMultinomial NB - 'no' yönünde en güçlü 12 özellik:")
for i in order_mnb[:12]:
    print(f"  {feature_names[i]:20s} {diff_mnb[i]:.3f}")

## Bu bölümü kaydet
Bir sonraki bölümün bu noktadan devam edebilmesi için tüm oturum (değişkenler, modeller, fonksiyonlar) diske kaydedilir.

In [ ]:
import dill
dill.dump_session('checkpoint_5.pkl')
print('Oturum checkpoint_5.pkl olarak kaydedildi.')